In [1]:
import PitchEvaluations.EvaluationFunctions as eva
import StockPrices.StockFetcher as Fetcher
import pandas as pd
import datetime as dt

In [2]:
t = pd.read_csv("ProcessedPortfolio.csv")
portfolio = t.groupby("Ticker").head(1).reset_index()
portfolio["Weight in Portofolio "] = t.groupby("Ticker").mean().reset_index()["Weight in Portofolio "]

options = pd.read_csv("constituents.csv")
# remove stocks already in portfolio
options = options.set_index("Symbol").drop(portfolio["Ticker"], errors='ignore').reset_index()

## Select timeframe and stocks you want might want to suggest

In [3]:
end = dt.date.today() # Fetches stock values from today
start = end - dt.timedelta(days=0.1*365) # Up until x days in the past

market_ticker = "^GSPC" # For beta calculation

# Reduce options if looking for something specific
options = options[options["Sector"]=="Industrials"].head(5)
options_tickers = options["Symbol"].tolist() # Make sure ticker column is called Ticker
options_tickers

['MMM', 'AOS', 'ALK', 'ALLE', 'AAL']

In [4]:
updated_tickers = Fetcher.prepare_tickers(portfolio["Ticker"].tolist()+[market_ticker], 
                                          portfolio["Exchange"].tolist()+[""])

portfolio_closes, nf = Fetcher.get_stock_prices(updated_tickers, start, end, True)
print("Could not find stocks:", nf)

Could not find stocks: []


In [5]:
optional_closes, nf = Fetcher.get_stock_prices(Fetcher.prepare_tickers(options_tickers, 
                                                              [""] * len(options_tickers)), start, end, True)
optional_closes = optional_closes.drop(nf, axis=1)
options_tickers = list(set(options_tickers) - set(nf))
print("Could not find stocks:", nf)

Could not find stocks: []


# Evaluation metrics to compute
Evaluation criteria: LogRetCor, Sharpe, Sortino, Beta
Default is True for all

In [6]:
closes = pd.concat([portfolio_closes, optional_closes], axis=1)

r = eva.evaluate(closes=closes, weights=portfolio['Weight in Portofolio '], tickers=options_tickers, market_ticker=market_ticker)


# Filter out the stocks that don't fit criteria

In [7]:
r

Symbols,MMM,ALLE,AOS,ALK,AAL
Symbols,,,,,
Beta,0.727760,1.303042,1.055253,0.881677,0.842293
Sortino ratio,1.301997,0.600082,0.679985,0.380188,0.555460
Sharpe ratio,0.462796,0.272051,0.314127,0.239473,0.339344
Portfolio,0.663734,0.791788,0.802848,0.709582,0.606391
AD.AS,-0.194773,-0.136176,-0.245998,-0.099683,-0.004421
KER.PA,0.059911,0.193188,0.268422,0.127615,0.167444
LOW,0.625115,0.822903,0.793410,0.732970,0.538027
WBD,0.465930,0.323448,0.234716,0.294112,0.240581
MA,0.757938,0.751240,0.659083,0.658917,0.545130
